# W3 Day1 (04/21 周一): GNN 原理 ★

## 学习目标
- **理论 (1-1.5h)**: 消息传递范式统一理解; GCN 空域直觉; GAT 注意力机制; GraphSAGE 采样策略; 异构图; 过平滑问题
- **实践 (2h)**: PyG 实现 GCN/GAT/GraphSAGE + Cora 三模型对比
- **产出物**: 三模型对比 notebook (Cora 上的 accuracy/收敛速度/参数量)

## 核心问题 (面试高频)
1. 消息传递范式是什么？GCN/GAT/SAGE 如何统一在这个框架下？
2. GCN 的空域理解：为什么是归一化的邻接矩阵乘特征？
3. GAT 的注意力权重怎么算？和 Transformer 的 self-attention 有什么区别？
4. GraphSAGE 为什么要采样？mini-batch 训练怎么做的？
5. 过平滑 (over-smoothing) 是什么？层数加深为什么反而变差？
6. 异构图和同构图的区别？RGCN 怎么处理多类型边？

---

## 目录
1. [消息传递范式](#1)
2. [GCN: 图卷积网络](#2)
3. [GAT: 图注意力网络](#3)
4. [GraphSAGE: 采样与聚合](#4)
5. [异构图与 RGCN](#5)
6. [过平滑问题](#6)
7. [PyG 实战: Cora 三模型对比](#7)
8. [总结 & 思考题](#8)

---
## 1. 消息传递范式 (Message Passing Paradigm) <a id='1'></a>

### 1.1 统一框架

几乎所有 GNN 都可以统一为 **消息传递** 框架:

$$h_v^{(l+1)} = \text{UPDATE}^{(l)}\left(h_v^{(l)},\; \text{AGGREGATE}^{(l)}\left(\{m_{u \to v}^{(l)} : u \in \mathcal{N}(v)\}\right)\right)$$

其中:
- $h_v^{(l)}$: 节点 $v$ 在第 $l$ 层的特征
- $\mathcal{N}(v)$: 节点 $v$ 的邻居集合
- $m_{u \to v}^{(l)}$: 从 $u$ 传到 $v$ 的消息 (message)
- AGGREGATE: 聚合邻居消息 (sum / mean / max / attention-weighted)
- UPDATE: 更新节点特征 (MLP / GRU / concat + linear)

### 1.2 三步流程

```
Step 1: MESSAGE  — 每条边 (u→v) 生成消息 m = MSG(h_u, h_v, e_uv)
Step 2: AGGREGATE — 每个节点聚合来自邻居的消息: a_v = AGG({m})
Step 3: UPDATE   — 更新节点特征: h_v' = UPD(h_v, a_v)
```

### 1.3 三种模型的对比

| | GCN | GAT | GraphSAGE |
|---|---|---|---|
| **消息** | $\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2} X W$ | $\alpha_{uv} W h_u$ | 采样邻居后 MSG(h_u) |
| **聚合** | 加权求和 (固定权重) | 注意力加权求和 | mean / LSTM / max |
| **更新** | 线性变换 + 激活 | 拼接 + 激活 | 拼接自身 + 邻居聚合 → MLP |
| **采样** | 全图 | 全图 | 邻居采样 (mini-batch) |

In [ ]:
import os
import time
import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# PyG
try:
    from torch_geometric.datasets import Planetoid
    from torch_geometric.nn import GCNConv, GATConv, SAGEConv
    from torch_geometric.utils import to_networkx
    HAS_PYG = True
except ImportError:
    HAS_PYG = False
    print("PyG 未安装，将用手写实现")

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"PyG: {'available' if HAS_PYG else 'not installed'}")
print("=" * 60)

---
## 2. GCN: 图卷积网络 <a id='2'></a>

### 2.1 空域直觉

GCN 的核心思想: **每个节点的新特征 = 自身特征 + 邻居特征的加权平均**

单层 GCN 的公式:
$$H^{(l+1)} = \sigma\left(\tilde{D}^{-\frac{1}{2}} \tilde{A} \tilde{D}^{-\frac{1}{2}} H^{(l)} W^{(l)}\right)$$

其中:
- $\tilde{A} = A + I$: 加自环的邻接矩阵
- $\tilde{D}$: $\tilde{A}$ 的度矩阵
- $\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$: **对称归一化**，防止度大的节点主导

### 2.2 为什么要做对称归一化？

不归一化: 节点 $v$ 的度越大 → 聚合时求和值越大 → 数值不稳定、训练困难

对称归一化的效果:
```
节点 v (度=3) 收到来自邻居 u1, u2, u3 的消息:
  权重 = 1/sqrt(d_v * d_u)  对每条边
  ≈ 每个邻居贡献 1/d_v 的量级
  → 无论度多大，聚合后的量级稳定
```

### 2.3 GCN 的局限

- **固定权重**: 每条边的权重只取决于度，不取决于节点内容
- **转导学习 (transductive)**: 标准 GCN 需要整张图，不能泛化到新节点
- **过平滑**: 层数多了以后，所有节点的表示趋于相同

In [ ]:
# ============ 手写 GCN 层 (不依赖 PyG) ============

class GCNLayer(nn.Module):
    """
    手写 GCN 层
    
    数学: H' = σ(D̃^{-1/2} Ã D̃^{-1/2} H W)
    
    输入:  x (N, F_in), edge_index (2, E)
    输出:  out (N, F_out)
    """
    def __init__(self, in_channels, out_channels, bias=True):
        super().__init__()
        self.linear = nn.Linear(in_channels, out_channels, bias=bias)
        nn.init.xavier_uniform_(self.linear.weight)
    
    def forward(self, x, edge_index):
        N = x.size(0)
        src, dst = edge_index  # src→dst 的边
        
        # 构建 Ã = A + I (加自环)
        # 度矩阵 D̃
        deg = torch.zeros(N, device=x.device)
        deg.scatter_add_(0, dst, torch.ones(edge_index.size(1), device=x.device))
        deg += 1  # 加自环
        
        # D̃^{-1/2}
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        
        # 边权重: 1/sqrt(d_u * d_v)
        norm = deg_inv_sqrt[src] * deg_inv_sqrt[dst]
        
        # 消息传递: 邻居聚合
        # m = norm * x[src]  (每条边的消息)
        msg = x[src] * norm.unsqueeze(-1)  # (E, F)
        
        # 聚合 (scatter_add)
        out = torch.zeros_like(x)
        out.scatter_add_(0, dst.unsqueeze(-1).expand_as(msg), msg)
        
        # 加自环: x / sqrt(d_v) ≈ 已包含在 deg 中
        # 实际上自环消息: x * deg_inv_sqrt[v]^2 = x / d_v
        # 但标准 GCN 公式里自环已经包含在 Ã 中，上面的 deg 已经 +1 了
        # 所以自环消息已经通过 deg_inv_sqrt 处理了
        
        # 线性变换 + 激活
        out = self.linear(out)
        return out


# 验证
gcn = GCNLayer(16, 32)
x = torch.randn(100, 16)
# 随机图: 200 条边
edge_index = torch.stack([
    torch.randint(0, 100, (200,)),
    torch.randint(0, 100, (200,))
])

out = gcn(x, edge_index)
print(f"GCN 手写层验证:")
print(f"  输入: x{tuple(x.shape)}, edges{tuple(edge_index.shape)}")
print(f"  输出: {tuple(out.shape)}")
print(f"  参数量: {sum(p.numel() for p in gcn.parameters()):,}")

---
## 3. GAT: 图注意力网络 <a id='3'></a>

### 3.1 核心思想

GCN 的边权重只取决于节点度 (结构信息)，**不考虑节点特征**。

GAT: 用 **注意力机制** 动态计算每条边的权重:

$$\alpha_{ij} = \text{softmax}_j\left(\text{LeakyReLU}\left(\mathbf{a}^T [W h_i \| W h_j]\right)\right)$$

$$h_i' = \sigma\left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij} W h_j\right)$$

### 3.2 与 Transformer self-attention 的区别

| | GAT | Transformer |
|---|---|---|
| **注意力范围** | 一阶邻居 $\mathcal{N}(i)$ | 所有位置 (全局) |
| **计算方式** | $\mathbf{a}^T [Wh_i \| Wh_j]$ (拼接) | $\frac{Q K^T}{\sqrt{d_k}}$ (点积) |
| **复杂度** | O(E) (边数) | O(N²) (节点数平方) |
| **位置编码** | 无 (图结构隐式编码) | 需要 (正弦/学习/RoPE) |
| **mask** | 不需要 (邻居天然稀疏) | causal mask (序列任务) |

### 3.3 多头注意力

和 Transformer 一样，GAT 也用多头:
- **中间层**: $K$ 个头**拼接** → $h_i' = \|_{k=1}^K \sigma(\sum \alpha_{ij}^k W^k h_j)$
- **输出层**: $K$ 个头**平均** → $h_i' = \sigma(\frac{1}{K} \sum_k \sum \alpha_{ij}^k W^k h_j)$

In [ ]:
# ============ 手写 GAT 层 ============

class GATLayer(nn.Module):
    """
    手写 Graph Attention Layer
    
    α_ij = softmax_j(LeakyReLU(a^T [Wh_i || Wh_j]))
    h_i' = σ(Σ α_ij W h_j)
    """
    def __init__(self, in_channels, out_channels, n_heads=4, concat=True, dropout=0.6):
        super().__init__()
        self.n_heads = n_heads
        self.out_channels = out_channels
        self.concat = concat
        self.head_dim = out_channels // n_heads
        assert out_channels % n_heads == 0
        
        # W: 共享的线性变换
        self.W = nn.Linear(in_channels, out_channels, bias=False)
        # 注意力向量 a: 每个头有 2 * head_dim 维
        self.a = nn.Parameter(torch.empty(n_heads, 2 * self.head_dim))
        nn.init.xavier_uniform_(self.a)
        
        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index):
        N = x.size(0)
        src, dst = edge_index
        
        # 线性变换: (N, F) → (N, n_heads, head_dim)
        h = self.W(x).view(N, self.n_heads, self.head_dim)
        
        # 计算注意力分数: a^T [h_i || h_j]
        h_i = h[dst]  # (E, n_heads, head_dim)
        h_j = h[src]  # (E, n_heads, head_dim)
        
        # 拼接: (E, n_heads, 2 * head_dim)
        cat = torch.cat([h_i, h_j], dim=-1)
        
        # 注意力分数: (E, n_heads)
        e = (cat * self.a.unsqueeze(0)).sum(dim=-1)
        e = self.leaky_relu(e)
        
        # Softmax (按目标节点 dst 分组)
        # 需要 scatter softmax
        alpha = self._scatter_softmax(e, dst, N)
        alpha = self.dropout(alpha)
        
        # 聚合: Σ α_ij * h_j
        # msg = alpha * h_j: (E, n_heads, head_dim)
        msg = h_j * alpha.unsqueeze(-1)
        
        # scatter add
        if self.concat:
            out = torch.zeros(N, self.n_heads, self.head_dim, device=x.device)
        else:
            out = torch.zeros(N, self.n_heads, self.head_dim, device=x.device)
        out.scatter_add_(0, dst.unsqueeze(-1).unsqueeze(-1).expand_as(msg), msg)
        
        if self.concat:
            out = out.view(N, -1)  # (N, out_channels)
        else:
            out = out.mean(dim=1)  # (N, head_dim)
        
        return out
    
    def _scatter_softmax(self, e, idx, N):
        """按 idx 分组的 softmax"""
        # 减去 max 防止溢出
        max_val = torch.full((N, self.n_heads), -1e9, device=e.device)
        max_val.scatter_reduce_(0, idx.unsqueeze(-1).expand_as(e), e, reduce='amax')
        e = e - max_val[idx]
        
        exp_e = e.exp()
        sum_exp = torch.zeros(N, self.n_heads, device=e.device)
        sum_exp.scatter_add_(0, idx.unsqueeze(-1).expand_as(exp_e), exp_e)
        
        return exp_e / (sum_exp[idx] + 1e-10)


# 验证
gat = GATLayer(16, 32, n_heads=4)
x = torch.randn(100, 16)
edge_index = torch.stack([
    torch.randint(0, 100, (300,)),
    torch.randint(0, 100, (300,))
])

out = gat(x, edge_index)
print(f"GAT 手写层验证:")
print(f"  输入: x{tuple(x.shape)}, edges{tuple(edge_index.shape)}")
print(f"  输出: {tuple(out.shape)}")
print(f"  参数量: {sum(p.numel() for p in gat.parameters()):,}")
print(f"  注意力头数: {gat.n_heads}, 每头维度: {gat.head_dim}")

---
## 4. GraphSAGE: 采样与聚合 <a id='4'></a>

### 4.1 为什么需要采样？

GCN/GAT 的问题: **需要整张图** 在内存中 → 大图 (百万节点) 不可行

GraphSAGE 的解决方案:
1. **邻居采样**: 每层只采样固定数量的邻居 (如 15 个)
2. **归纳学习 (inductive)**: 学习的是聚合函数，不是节点嵌入 → 可泛化到新节点
3. **Mini-batch 训练**: 每个 batch 只计算一个子图

### 4.2 算法流程

```
对每个目标节点 v:
  Layer 1: 采样 N(v) 中的 k1 个邻居 → 聚合
  Layer 2: 对每个邻居再采样 k2 个邻居 → 聚合
  ...
  最终: h_v' = CONCAT(h_v, AGG(neighbors))  → W
```

### 4.3 三种聚合器

| 聚合器 | 公式 | 特点 |
|---|---|---|
| **Mean** | $\frac{1}{|\mathcal{N}|}\sum h_u$ | 简单、稳定、默认选择 |
| **LSTM** | LSTM([h_u : u ∈ perm(N)]) | 表达力强，但需要邻居排序 |
| **Max/Pool** | $\max(\{\text{MLP}(h_u)\})$ | 捕捉最显著特征 |

### 4.4 采样数量的选择

论文默认: 第 1 层采 25，第 2 层采 10 → 25×10 = 250 个二阶邻居

实际工程: 根据图的度分布调整。度很大的图可以少采，度小的图需要多采。

In [ ]:
# ============ 手写 GraphSAGE 层 ============

class SAGELayer(nn.Module):
    """
    手写 GraphSAGE 层 (Mean 聚合器)
    
    h_v' = σ(W · CONCAT(h_v, MEAN({h_u : u ∈ N(v)})))
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # 注意: 输入维度是 2 * in_channels (因为拼接自身+邻居)
        self.linear = nn.Linear(2 * in_channels, out_channels)
        nn.init.xavier_uniform_(self.linear.weight)
    
    def forward(self, x, edge_index):
        N = x.size(0)
        src, dst = edge_index
        
        # 邻居聚合 (mean)
        agg = torch.zeros_like(x)
        count = torch.zeros(N, 1, device=x.device)
        
        agg.scatter_add_(0, dst.unsqueeze(-1).expand_as(x[src]), x[src])
        count.scatter_add_(0, dst.unsqueeze(-1), torch.ones(src.size(0), 1, device=x.device))
        count.clamp_(min=1)
        agg = agg / count
        
        # 拼接自身 + 邻居聚合
        out = torch.cat([x, agg], dim=-1)  # (N, 2*F)
        
        # 线性变换
        out = self.linear(out)
        return out


# 验证
sage = SAGELayer(16, 32)
x = torch.randn(100, 16)
edge_index = torch.stack([
    torch.randint(0, 100, (300,)),
    torch.randint(0, 100, (300,))
])

out = sage(x, edge_index)
print(f"GraphSAGE 手写层验证:")
print(f"  输入: x{tuple(x.shape)}, edges{tuple(edge_index.shape)}")
print(f"  输出: {tuple(out.shape)}")
print(f"  参数量: {sum(p.numel() for p in sage.parameters()):,}")
print(f"  注意: 输入是 2*16=32 维 (拼接自身+邻居)")

---
## 5. 异构图与 RGCN <a id='5'></a>

### 5.1 同构 vs 异构

| | 同构图 | 异构图 |
|---|---|---|
| **节点类型** | 一种 | 多种 (如 用户、商品、店铺) |
| **边类型** | 一种 | 多种 (如 购买、浏览、属于) |
| **例子** | Cora 论文引用图 | 知识图谱、推荐系统 |

### 5.2 RGCN (Relational GCN)

核心思想: **每种边类型有独立的权重矩阵**

$$h_v^{(l+1)} = \sigma\left(\sum_{r \in \mathcal{R}} \sum_{u \in \mathcal{N}_r(v)} \frac{1}{c_{v,r}} W_r^{(l)} h_u^{(l)} + W_0^{(l)} h_v^{(l)}\right)$$

其中:
- $\mathcal{R}$: 所有关系类型
- $W_r$: 关系 $r$ 的独立权重矩阵
- $c_{v,r}$: 归一化常数 (通常 = |N_r(v)|)
- $W_0$: 自环权重

### 5.3 参数爆炸问题

|关系类型多 → 权重矩阵多 → 参数量爆炸

解决方案:
- **基分解**: $W_r = \sum_{b=1}^B a_{rb} V_b$，共享 $B$ 个基矩阵
- **块对角**: $W_r = \text{diag}(W_{r1}, ..., W_{rB})$，减少参数

---
## 6. 过平滑问题 <a id='6'></a>

### 6.1 现象

GNN 层数增加 → 每个节点的感受野指数级增长 → 所有节点看到几乎相同的邻居 → **节点表示趋于一致**

```
Layer 1: 节点看 1-hop 邻居 (局部)
Layer 2: 节点看 2-hop 邻居 (开始重叠)
Layer 3: 节点看 3-hop 邻居 (大量重叠)
Layer 5: 几乎所有节点看到整张图 → 表示相同
```

### 6.2 量化度量

- **余弦相似度**: 节点对之间的平均余弦相似度随层数增加而趋近 1
- **分类准确率**: 先升后降 (2-3 层最优，超过后急剧下降)

### 6.3 缓解方法

| 方法 | 思路 | 复杂度 |
|---|---|---|
| **残差连接** | $h^{(l+1)} = h^{(l)} + GNN(h^{(l)})$ | 低 |
| **JK-Net** | 每层输出都参与最终聚合 | 中 |
| **DropEdge** | 训练时随机删边 | 低 |
| **PairNorm** | 归一化节点对之间的距离 | 低 |
| **GPRGNN** | 学习各层的贡献权重 | 中 |
| **DeeperGCN** | PreLN + 残差 + 虚拟节点 | 中 |

In [ ]:
# ============ 过平滑可视化 ============

# 用一个简单 GNN 堆叠不同层数，观察节点表示的相似度变化

class SimpleGCN(nn.Module):
    """简单的多层 GCN (用于过平滑实验)"""
    def __init__(self, in_dim, hidden_dim, out_dim, n_layers):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GCNLayer(in_dim, hidden_dim))
        for _ in range(n_layers - 2):
            self.convs.append(GCNLayer(hidden_dim, hidden_dim))
        self.convs.append(GCNLayer(hidden_dim, out_dim))
    
    def forward(self, x, edge_index):
        embeddings = []
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
            embeddings.append(x)
        return x, embeddings


# 生成一个随机图
N = 500
F_dim = 64
x = torch.randn(N, F_dim)
# 构建一个有社区结构的图 (两个社区)
edges_intra = []
for _ in range(800):
    c = np.random.choice([0, 1])
    u = np.random.randint(c * 250, (c + 1) * 250)
    v = np.random.randint(c * 250, (c + 1) * 250)
    edges_intra.append((u, v))
edges_inter = []
for _ in range(100):
    u = np.random.randint(0, 250)
    v = np.random.randint(250, 500)
    edges_inter.append((u, v))
edges = edges_intra + edges_inter
edge_index = torch.tensor(edges, dtype=torch.long).T

# 不同层数的 GNN
layer_counts = [1, 2, 3, 4, 5, 8, 16]
avg_similarities = []

for n_layers in layer_counts:
    model = SimpleGCN(F_dim, 64, 32, n_layers)
    model.eval()
    with torch.no_grad():
        out, _ = model(x, edge_index)
    # 计算节点对之间的平均余弦相似度 (采样 1000 对)
    idx1 = torch.randint(0, N, (1000,))
    idx2 = torch.randint(0, N, (1000,))
    cos_sim = F.cosine_similarity(out[idx1], out[idx2], dim=-1)
    avg_similarities.append(cos_sim.mean().item())

# 可视化
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(layer_counts, avg_similarities, 'o-', color='#c2553a', linewidth=2, markersize=8)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Number of GNN Layers')
ax.set_ylabel('Avg Cosine Similarity')
ax.set_title('Over-smoothing: Node Representations Become Indistinguishable')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("过平滑实验:")
for n, s in zip(layer_counts, avg_similarities):
    bar = '█' * int(s * 30)
    print(f"  {n:2d} layers: sim={s:.3f}  {bar}")
print(f"\n💡 3-4 层以后，节点表示趋同 → 分类性能下降")
print(f"💡 这就是为什么大多数 GNN 只用 2-3 层")

---
## 7. PyG 实战: Cora 三模型对比 <a id='7'></a>

### Cora 数据集

- **节点**: 2708 篇论文
- **边**: 5429 条引用关系
- **特征**: 1433 维 (bag-of-words)
- **类别**: 7 个领域 (Case_Based, Genetic_Algorithms, Neural_Networks, ...)
- **任务**: 节点分类 (transductive: 训练 140 / 验证 500 / 测试 1000)

In [ ]:
# ============ 加载 Cora 数据集 ============

if HAS_PYG:
    dataset = Planetoid(root='/tmp/Cora', name='Cora')
    data = dataset[0]
    print(f"Cora 数据集:")
    print(f"  节点: {data.num_nodes}")
    print(f"  边: {data.num_edges}")
    print(f"  特征维度: {dataset.num_features}")
    print(f"  类别数: {dataset.num_classes}")
    print(f"  训练/验证/测试: {data.train_mask.sum()}/{data.val_mask.sum()}/{data.test_mask.sum()}")
    
    # 度分布
    src, dst = data.edge_index
    deg = torch.zeros(data.num_nodes)
    deg.scatter_add_(0, dst, torch.ones(data.num_edges))
    print(f"  平均度: {deg.mean():.1f}")
    print(f"  最大度: {deg.max():.0f}")
else:
    print("PyG 未安装，跳过 Cora 加载")

In [ ]:
# ============ 定义三个模型 ============

class GCNModel(nn.Module):
    """2-layer GCN"""
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)
        self.dropout = dropout
    
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class GATModel(nn.Module):
    """2-layer GAT (8 heads → 1 head)"""
    def __init__(self, in_dim, hidden_dim, out_dim, heads=8, dropout=0.6):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim // heads, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_dim, out_dim, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout
    
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class SAGEModel(nn.Module):
    """2-layer GraphSAGE"""
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.dropout = dropout
    
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


print("三个模型定义完成:")
print(f"  GCN:  GCNConv → ReLU → GCNConv")
print(f"  GAT:  GATConv(8 heads) → ELU → GATConv(1 head)")
print(f"  SAGE: SAGEConv → ReLU → SAGEConv")

In [ ]:
# ============ 训练与评估 ============

def train_and_eval(model, data, lr=0.01, weight_decay=5e-4, epochs=200):
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    train_losses = []
    val_accs = []
    
    t0 = time.time()
    for epoch in range(epochs):
        # Train
        model.train()
        optimizer.zero_grad()
        out = model(data)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        
        # Eval
        model.eval()
        with torch.no_grad():
            pred = model(data).argmax(dim=1)
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            val_accs.append(val_acc)
    
    # Final test accuracy
    model.eval()
    with torch.no_grad():
        pred = model(data).argmax(dim=1)
        test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    
    elapsed = time.time() - t0
    n_params = sum(p.numel() for p in model.parameters())
    
    return {
        'train_losses': train_losses,
        'val_accs': val_accs,
        'test_acc': test_acc,
        'best_val_acc': max(val_accs),
        'n_params': n_params,
        'time': elapsed,
    }


if HAS_PYG:
    hidden_dim = 64
    epochs = 200
    
    results = {}
    for name, Model in [('GCN', GCNModel), ('GAT', GATModel), ('GraphSAGE', SAGEModel)]:
        print(f"\n训练 {name}...")
        model = Model(dataset.num_features, hidden_dim, dataset.num_classes)
        res = train_and_eval(model, data, epochs=epochs)
        results[name] = res
        print(f"  参数量: {res['n_params']:,}")
        print(f"  训练时间: {res['time']:.1f}s")
        print(f"  最佳验证准确率: {res['best_val_acc']:.4f}")
        print(f"  测试准确率: {res['test_acc']:.4f}")
else:
    print("跳过训练 (PyG 未安装)")

In [ ]:
# ============ 可视化对比 ============

if HAS_PYG:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    colors = {'GCN': '#5a6b4a', 'GAT': '#c2553a', 'GraphSAGE': '#2d2a26'}
    
    # 训练损失
    ax = axes[0]
    for name, res in results.items():
        ax.plot(res['train_losses'], label=name, color=colors[name], linewidth=1.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Training Loss')
    ax.set_title('Training Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 验证准确率
    ax = axes[1]
    for name, res in results.items():
        ax.plot(res['val_accs'], label=name, color=colors[name], linewidth=1.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation Accuracy')
    ax.set_title('Validation Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 测试准确率 + 参数量 柱状图
    ax = axes[2]
    names = list(results.keys())
    test_accs = [results[n]['test_acc'] for n in names]
    bar_colors = [colors[n] for n in names]
    bars = ax.bar(names, test_accs, color=bar_colors, alpha=0.8)
    for bar, acc in zip(bars, test_accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{acc:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylabel('Test Accuracy')
    ax.set_title('Test Accuracy Comparison')
    ax.set_ylim(0.7, 0.85)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # 汇总表
    print("\n" + "=" * 65)
    print(f"{'Model':<12} {'Params':>10} {'Best Val':>10} {'Test Acc':>10} {'Time':>8}")
    print("-" * 65)
    for name, res in results.items():
        print(f"{name:<12} {res['n_params']:>10,} {res['best_val_acc']:>10.4f} "
              f"{res['test_acc']:>10.4f} {res['time']:>7.1f}s")
    print("=" * 65)

In [ ]:
# ============ 注意力权重可视化 (GAT) ============

if HAS_PYG:
    # 提取 GAT 的注意力权重
    gat_model = GATModel(dataset.num_features, 64, dataset.num_classes).to(device)
    data_device = data.to(device)
    
    # 训练一下
    optimizer = torch.optim.Adam(gat_model.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(200):
        gat_model.train()
        optimizer.zero_grad()
        out = gat_model(data_device)
        loss = F.cross_entropy(out[data_device.train_mask], data_device.y[data_device.train_mask])
        loss.backward()
        optimizer.step()
    
    # 获取注意力权重
    gat_model.eval()
    with torch.no_grad():
        # 手动跑第一层获取 attention
        x = data_device.x
        x_drop = F.dropout(x, p=0.6, training=False)
        x1, (edge_index_out, attn_weights) = gat_model.conv1(x_drop, data_device.edge_index, return_attention_weights=True)
    
    # 注意力分布统计
    attn = attn_weights.cpu().numpy()
    print(f"GAT 注意力权重统计:")
    print(f"  总边数: {len(attn)}")
    print(f"  注意力均值: {attn.mean():.4f}")
    print(f"  注意力标准差: {attn.std():.4f}")
    print(f"  最大注意力: {attn.max():.4f}")
    print(f"  最小注意力: {attn.min():.4f}")
    
    # 直方图
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(attn, bins=50, color='#c2553a', alpha=0.7, edgecolor='white')
    ax.set_xlabel('Attention Weight')
    ax.set_ylabel('Count')
    ax.set_title('GAT Attention Weight Distribution (Layer 1)')
    ax.axvline(attn.mean(), color='#2d2a26', linestyle='--', label=f'mean={attn.mean():.4f}')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"\n💡 注意力分布越不均匀 → 模型越会选择性地关注重要邻居")
    print(f"💡 如果分布接近 uniform → 注意力机制没有发挥作用")

---
## 8. 总结 & 思考题 <a id='8'></a>

### 今日核心知识点

1. **消息传递范式**: MESSAGE → AGGREGATE → UPDATE，GCN/GAT/SAGE 都是特例
2. **GCN**: 对称归一化邻接矩阵，固定权重，简单高效
3. **GAT**: 注意力动态加权，表达力更强，但计算量更大
4. **GraphSAGE**: 邻居采样 + 归纳学习，支持 mini-batch 大图训练
5. **异构图**: RGCN 为每种边类型学习独立权重 (基分解/块对角解决参数爆炸)
6. **过平滑**: 层数加深 → 节点表示趋同 → 2-3 层最优

### 面试高频问题

1. **GCN 和 GAT 的本质区别？** GCN 的权重只取决于图结构 (度)，GAT 的权重同时考虑节点特征 (注意力)
2. **GAT 和 Transformer self-attention 的区别？** GAT 只在邻居上算注意力 (稀疏, O(E))，Transformer 在所有位置上算 (稠密, O(N²))
3. **GraphSAGE 为什么能处理新节点？** 它学习的是聚合函数 (参数)，不是节点嵌入 (查找表) → 归纳学习
4. **过平滑怎么缓解？** 残差连接、DropEdge、JK-Net、DeeperGCN
5. **为什么大多数 GNN 只用 2-3 层？** 过平滑 + 感受野指数增长，2-3 层覆盖 2-3 跳邻居已经够用
6. **异构图怎么处理？** RGCN: 每种关系类型一个权重矩阵; 参数太多时用基分解

### 与项目/简历的关联

- **GNN 图纸审查项目 (W3D2)**: GAT 注意力机制可以可视化哪些 CAD 图元被重点关注
- **知识图谱**: RGCN 是知识图谱嵌入的基础方法，与你博士的 Ontology+KG 工作直接相关
- **面试**: GNN 是图任务的基础，GCN vs GAT vs SAGE 的 trade-off 是高频考点

### 明日预习: GNN 进阶 + 图数据增强 ★

- DropEdge / 节点 Mask / 子图采样
- 半监督训练策略
- GNN 的局限性与改进方向

In [ ]:
print("=" * 60)
print("W3 Day1 完成!")
print("=" * 60)
if HAS_PYG:
    print(f"""
今日成果:
  ✅ 手写 GCN 层 (对称归一化 + scatter 实现)
  ✅ 手写 GAT 层 (多头注意力 + scatter softmax)
  ✅ 手写 GraphSAGE 层 (mean 聚合器 + concat 更新)
  ✅ 过平滑实验: 不同层数 GNN 的节点表示相似度对比
  ✅ Cora 三模型对比: GCN vs GAT vs GraphSAGE
  ✅ GAT 注意力权重可视化

Cora 结果:
  GCN:       test_acc = {results['GCN']['test_acc']:.4f}, params = {results['GCN']['n_params']:,}
  GAT:       test_acc = {results['GAT']['test_acc']:.4f}, params = {results['GAT']['n_params']:,}
  GraphSAGE: test_acc = {results['GraphSAGE']['test_acc']:.4f}, params = {results['GraphSAGE']['n_params']:,}
""")
else:
    print("\n安装 PyG 后可运行完整实验: pip install torch-geometric")